<a href="https://colab.research.google.com/github/SahityaKotla123/flyrank-ml/blob/main/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SahityaKotla123/flyrank-ml/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
*Provisional lane: Lane 2 — Refresh / Content Opportunity Scoring.**

The question I want to answer: *of the content a client already has, which pages should a
reviewer look at first — and for what reason?* I'm picking this lane over the others because
the starter pipeline already shows the shape of the payoff (see the numbers below): a learned
ranking beats a hand-written rule by a wide margin on precision@50, which is exactly the metric
a capacity-constrained review team cares about. It also gives me a natural growth path I can
start on the 30k-row starter CSV this week, then move to the warehouse's daily fact table later
to build a real future-window label (decline/recovery over the next 30 days) instead of the
starter's same-window proxy label. Ranking Signal Analysis (Lane 1) and CTR/Engagement Scoring
(Lane 4) are close cousins of this. I may fold pieces of their signal work in as features and
reason codes rather than switching lanes outright. I'm not choosing Lane 3 (clustering) because
I want a decision that produces an ordered action list, not a set of descriptive groups, and I'm
not going freestyle because this lane already matches the decision I actually care about.


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
## 2. The question: decision, action, cost of a wrong call

**Decision this improves:** which of a client's existing content pages a content reviewer
should open and work on first, out of far more candidates than they have hours for.

**Who acts, and what do they do:** a content strategist/reviewer at a FlyRank client, working
through a capacity-limited queue (they can realistically review something like 20-50 pages a
sprint, not thousands). Given my ranked list plus a reason code, they open the top pages first
and decide whether to refresh, expand, protect, or leave a page alone.

**Cost of a wrong call, in both directions:**
- *False positive* (I rank a fine page as "review me first"): wastes a reviewer's limited hours
  on a page that didn't need attention, and pushes an actually-declining page further down the
  queue — an opportunity cost, not just wasted time.
- *False negative* (a genuinely declining, high-demand page never surfaces near the top): the
  client keeps losing visibility/clicks on a page that mattered, and nobody notices until the
  drop is much larger. This is the more expensive mistake, since demand (impressions) is already
  flowing to that URL and every day of delay is compounding loss.

Because reviewer time is the scarce resource, the metric that matches this decision is
**precision@K** (K = the number of pages a team can actually act on per cycle), not overall
accuracy — I'd rather be right about the top 50 than "correct on average" across 30,000 rows.

**Why data or ML helps at all:** the signals that separate "review this now" from "leave it
alone" are not one clean rule — traffic trend, current visibility, position, freshness, content
depth, and engagement all interact, and their relative importance likely differs by content type
and client. A single if-statement (e.g. "flag anything not updated in 180 days") already exists
in the starter baseline and only captures a handful of pages (see the numbers below) while
missing most of the plausible opportunity — that's a sign the pattern is real but too tangled for
a hand-written rule alone, which is exactly where a learned model earns its place over a plain
rule.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [ ]:
!git clone https://github.com/SahityaKotla123/flyrank-ml.git

Cloning into 'flyrank-ml'...
remote: Enumerating objects: 117, done.
remote: Counting objects: 100% (117/117), done.
remote: Compressing objects: 100% (73/73), done.
remote: Total 117 (delta 33), reused 96 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (117/117), 1.84 MiB | 2.06 MiB/s, done.
Resolving deltas: 100% (33/33), done.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("flyrank-ml/data/raw/content_refresh_anonymized.csv")

n_rows = len(df)
n_clients = df["client_id"].nunique()
print(f"Starter dataset: {n_rows:,} rows x {df.shape[1]} columns across {n_clients} clients")

# Number 1: how big is the "declining with real demand" pool -- the population
# a refresh-scoring lane would actually try to prioritize inside.
declining_with_demand = df[(df["trend_direction"] == "down") & (df["impressions_90d"] >= 100)]
print(
    f"Rows flagged 'down' trend AND >=100 impressions/90d: {len(declining_with_demand):,} "
    f"({len(declining_with_demand) / n_rows:.1%} of all rows)"
)

# Number 2: a hand-written rule ("not updated in 180+ days, still gets real traffic")
# only catches a tiny sliver -- evidence a plain if-statement isn't enough on its own.
stale_visible = df[(df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)]
print(
    f"Rows matching the simple 'stale but visible' rule: {len(stale_visible):,} "
    f"({len(stale_visible) / n_rows:.1%} of all rows) -- too few to be the whole story"
)

# Number 3: how much of the review queue also has a CTR problem worth a reason code
low_ctr_visible = df[
    (df["impressions_90d"] >= 500) & (df["avg_position"] > 0)
    & (df["avg_position"] <= 20) & (df["ctr"] < 0.5)
]
print(
    f"Rows with real visibility (page 1-2, 500+ impressions) but CTR < 0.5%: "
    f"{len(low_ctr_visible):,} ({len(low_ctr_visible) / n_rows:.1%} of all rows)"
)

print()
print("trend_direction breakdown:")
print(df["trend_direction"].value_counts())

Starter dataset: 30,000 rows x 44 columns across 32 clients
Rows flagged 'down' trend AND >=100 impressions/90d: 13,152 (43.8% of all rows)
Rows matching the simple 'stale but visible' rule: 17 (0.1% of all rows) -- too few to be the whole story
Rows with real visibility (page 1-2, 500+ impressions) but CTR < 0.5%: 9,759 (32.5% of all rows)

trend_direction breakdown:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
## 4. Careful words: what I can and can't claim

**What I will be able to say:**
- *Observed*: which content-level signals (trend, position, freshness, depth, engagement) are
  associated with pages that later show decline or recovery, measured on held-out clients.
- *Directional*: that a learned ranking model concentrates true positives near the top of a
  review queue more effectively than a hand-written rule, on this dataset and this label.
- *Decision-support*: a ranked, reason-coded list a human reviewer can sanity-check and choose to
  act on — not an automated decision.

**What I will never claim:**
- That refreshing a page *caused* a recovery — I have no experiment or causal design, only
  observational data. At most I can say a page was flagged, and later note whether it moved.
- That I predicted anything about Google's ranking algorithm or discovered a ranking factor.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.